# Data Engineering Excercise — Versi pandas — Kunci Jawaban
Notebook ini berisi solusi lengkap dari `DE_pandas_excercise.ipynb`.
Setiap soal ditampilkan bersama data contoh, solusinya, hasilnya, dan hasil 3 test case.


## Persiapan

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [ ]:
import math

# Menjalankan sekumpulan test case terhadap fungsi answer.
# cases = list of (judul, argumen fungsi sebagai tuple, DataFrame yang diharapkan).
# Urutan baris, urutan kolom, index, dan tipe angka tidak diperhitungkan.
def run_tests(answer_fn, cases):
    def clean(value):
        if pd.isna(value):
            return None
        if isinstance(value, np.integer):
            value = int(value)
        elif isinstance(value, np.floating):
            value = float(value)
        if isinstance(value, float) and value.is_integer():
            return int(value)
        return value

    def norm(frame):
        columns = sorted(str(name) for name in frame.columns)
        rows = frame.rename(columns=str)[columns]
        rows = [tuple(clean(v) for v in row)
                for row in rows.itertuples(index=False, name=None)]
        return columns, sorted(rows, key=repr)

    passed = 0
    for number, (title, args, expected) in enumerate(cases, 1):
        try:
            result = answer_fn(*args)
        except Exception as error:
            print(f"[{number}] {title}: ERROR -> {type(error).__name__}: {error}")
            continue

        if not isinstance(result, pd.DataFrame):
            print(f"[{number}] {title}: FAIL")
            print(f"    answer harus mengembalikan DataFrame, "
                  f"bukan {type(result).__name__}")
            continue

        if norm(result) == norm(expected):
            print(f"[{number}] {title}: PASS")
            passed += 1
        else:
            print(f"[{number}] {title}: FAIL")
            print("    diharapkan :")
            print(expected.to_string(index=False))
            print("    hasil anda :")
            print(result.to_string(index=False))
    print(f"\n{passed}/{len(cases)} test case lulus")

## Soal 1
Diberikan sebuah tabel `Products` yang memiliki kolom `low_fats` dan `recyclable` berisi `'Y'`/`'N'`. Tunjukkan `product_id` untuk produk yang bersifat low fats dan recyclable.


### Data

In [ ]:
products = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5],
    "low_fats":   ["Y", "Y", "N", "Y", "N"],
    "recyclable": ["N", "Y", "Y", "Y", "N"],
})

display(products)

### Solusi

In [ ]:
def answer(products):
    # `and` tidak bisa dipakai pada Series -> gunakan & dengan tanda kurung
    mask = (products["low_fats"] == "Y") & (products["recyclable"] == "Y")
    return products.loc[mask, ["product_id"]]

### Hasil

In [ ]:
hasil = answer(products)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 1
cases = [
    ("Semua produk memenuhi syarat", (
        pd.DataFrame([
            {"product_id": 10, "low_fats": 'Y', "recyclable": 'Y'},
            {"product_id": 11, "low_fats": 'Y', "recyclable": 'Y'},
        ]),
    ), pd.DataFrame([
        {"product_id": 10},
        {"product_id": 11},
    ])),

    ("Tidak ada yang memenuhi syarat", (
        pd.DataFrame([
            {"product_id": 1, "low_fats": 'Y', "recyclable": 'N'},
            {"product_id": 2, "low_fats": 'N', "recyclable": 'Y'},
            {"product_id": 3, "low_fats": 'N', "recyclable": 'N'},
        ]),
    ), pd.DataFrame(columns=['product_id'])),

    ("Tabel kosong", (
        pd.DataFrame({"product_id": pd.Series(dtype="int64"), "low_fats": pd.Series(dtype="str"), "recyclable": pd.Series(dtype="str")}),
    ), pd.DataFrame(columns=['product_id'])),

]

run_tests(answer, cases)

## Soal 2
Diberikan sebuah tabel `Customer` dengan kolom `id`, `name`, dan `refree_id`. Tunjukkan `name` pelanggan yang tidak direferensikan oleh pelanggan dengan `id = 2`. Pelanggan tanpa perekomendasi (`referee_id` NULL) juga terhitung.


### Data

In [ ]:
customer = pd.DataFrame({
    "id":         [1, 2, 3, 4, 5, 6],
    "name":       ["Sari", "Bima", "Rani", "Dewi", "Agus", "Tono"],
    "referee_id": [np.nan, np.nan, 2.0, 3.0, 2.0, np.nan],
})

display(customer)

### Solusi

In [ ]:
def answer(customer):
    # NaN != 2 menghasilkan True, tetapi tulis .isna() secara eksplisit
    # supaya niatnya terbaca dan tidak bergantung pada kebetulan
    mask = (customer["referee_id"] != 2) | customer["referee_id"].isna()
    return customer.loc[mask, ["name"]]

### Hasil

In [ ]:
hasil = answer(customer)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 2
cases = [
    ("Semua referee_id NULL", (
        pd.DataFrame([
            {"id": 1, "name": 'Ani', "referee_id": None},
            {"id": 2, "name": 'Budi', "referee_id": np.nan},
        ]),
    ), pd.DataFrame([
        {"name": 'Ani'},
        {"name": 'Budi'},
    ])),

    ("Semua direferensikan pelanggan 2", (
        pd.DataFrame([
            {"id": 3, "name": 'Cita', "referee_id": 2},
            {"id": 4, "name": 'Deni', "referee_id": 2.0},
        ]),
    ), pd.DataFrame(columns=['name'])),

    ("Campuran NULL, 2, dan bukan 2", (
        pd.DataFrame([
            {"id": 1, "name": 'Ani', "referee_id": None},
            {"id": 2, "name": 'Budi', "referee_id": 2},
            {"id": 3, "name": 'Cita', "referee_id": 3.0},
            {"id": 4, "name": 'Deni', "referee_id": 2.0},
        ]),
    ), pd.DataFrame([
        {"name": 'Ani'},
        {"name": 'Cita'},
    ])),

]

run_tests(answer, cases)

## Soal 3
Diberikan sebuah tabel `Tweets` dengan kolom `tweet_id` dan `content`. Tunjukkan `tweet_id` dengan panjang `content` lebih dari 15 karakter.


### Data

In [ ]:
tweets = pd.DataFrame({
    "tweet_id": [1, 2, 3, 4],
    "content": [
        "Halo dunia",
        "Belajar pandas itu menyenangkan sekali",
        "Data engineering",
        "Singkat",
    ],
})

display(tweets)

### Solusi

In [ ]:
def answer(tweets):
    return tweets.loc[tweets["content"].str.len() > 15, ["tweet_id"]]

### Hasil

In [ ]:
hasil = answer(tweets)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 3
cases = [
    ("Terdapat tweet dengan 16 karakter", (
        pd.DataFrame([
            {"tweet_id": 1, "content": '123456789012345'},
            {"tweet_id": 2, "content": '1234567890123456'},
        ]),
    ), pd.DataFrame([
        {"tweet_id": 2},
    ])),

    ("Semua konten pendek", (
        pd.DataFrame([
            {"tweet_id": 1, "content": 'Halo'},
            {"tweet_id": 2, "content": ''},
        ]),
    ), pd.DataFrame(columns=['tweet_id'])),

    ("Semua konten panjang", (
        pd.DataFrame([
            {"tweet_id": 7, "content": 'Data engineering itu seru'},
            {"tweet_id": 8, "content": 'Selamat pagi semuanya'},
        ]),
    ), pd.DataFrame([
        {"tweet_id": 7},
        {"tweet_id": 8},
    ])),

]

run_tests(answer, cases)

## Soal 4
Diberikan dua tabel yaitu `Visits` dengan kolom `visit_id` dan `customer_id`, serta `Transactions` dengan kolom `transaction_id`, `visit_id`, dan `amount`. Tampilkan dan hitung jumlah pelanggan kunjungan dari pelanggan yang berkunjung tanpa melakukan transaksi sama sekali.


### Data

In [ ]:
visits = pd.DataFrame({
    "visit_id":    [1, 2, 4, 5, 5, 6, 7, 8],
    "customer_id": [23, 9, 30, 54, 54, 96, 54, 54],
})
transactions = pd.DataFrame({
    "transaction_id": [2, 3, 9, 12],
    "visit_id":       [5, 5, 5, 1],
    "amount":         [310, 300, 200, 910],
})

display(visits)
display(transactions)

### Solusi

In [ ]:
def answer(visits, transactions):
    tanpa_transaksi = visits[~visits["visit_id"].isin(transactions["visit_id"])]

    return (tanpa_transaksi.groupby("customer_id", as_index=False)
                           .size()
                           .rename(columns={"size": "count_no_trans"}))

### Hasil

In [ ]:
hasil = answer(visits, transactions)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 4
cases = [
    ("Semua kunjungan ada transaksinya", (
        pd.DataFrame([
            {"visit_id": 1, "customer_id": 10},
            {"visit_id": 2, "customer_id": 20},
        ]),
        pd.DataFrame([
            {"transaction_id": 1, "visit_id": 1, "amount": 100},
            {"transaction_id": 2, "visit_id": 2, "amount": 200},
        ]),
    ), pd.DataFrame(columns=['customer_id', 'count_no_trans'])),

    ("Tidak ada transaksi sama sekali", (
        pd.DataFrame([
            {"visit_id": 1, "customer_id": 10},
            {"visit_id": 2, "customer_id": 10},
            {"visit_id": 3, "customer_id": 20},
        ]),
        pd.DataFrame({"transaction_id": pd.Series(dtype="int64"), "visit_id": pd.Series(dtype="int64"), "amount": pd.Series(dtype="int64")}),
    ), pd.DataFrame([
        {"customer_id": 10, "count_no_trans": 2},
        {"customer_id": 20, "count_no_trans": 1},
    ])),

    ("Satu pelanggan berkunjung dengan transaksi campuran", (
        pd.DataFrame([
            {"visit_id": 1, "customer_id": 7},
            {"visit_id": 2, "customer_id": 7},
            {"visit_id": 3, "customer_id": 7},
        ]),
        pd.DataFrame([
            {"transaction_id": 1, "visit_id": 2, "amount": 50},
        ]),
    ), pd.DataFrame([
        {"customer_id": 7, "count_no_trans": 2},
    ])),

]

run_tests(answer, cases)

## Soal 5
Diberikan dua tabel, yaitu `Employee(empId, name, supervisor, salary)` dengan kolom `empId`, `name`, `supervisor`, dan `salary`, serta `Bonus` dengan kolom `empId` dan `bonus`. Tampilkan nama dan bonus karyawan yang bonusnya di bawah 1000 atau tidak memiliki bonus sama sekali.


### Data

In [ ]:
employee = pd.DataFrame({
    "empId":      [3, 1, 2, 4],
    "name":       ["Brad", "John", "Dan", "Thomas"],
    "supervisor": [np.nan, 3.0, 3.0, 2.0],
    "salary":     [4000, 1000, 2000, 4000],
})
bonus = pd.DataFrame({
    "empId": [2, 4],
    "bonus": [500, 2000],
})

display(employee)
display(bonus)

### Solusi

In [ ]:
def answer(employee, bonus):
    gab = employee.merge(bonus, on="empId", how="left")

    # NaN < 1000 bernilai False, sehingga .isna() harus ditambahkan
    mask = (gab["bonus"] < 1000) | gab["bonus"].isna()
    return gab.loc[mask, ["name", "bonus"]]

### Hasil

In [ ]:
hasil = answer(employee, bonus)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 5
cases = [
    ("Semua bonus di atas 1000", (
        pd.DataFrame([
            {"empId": 1, "name": 'Ana', "supervisor": None, "salary": 3000},
            {"empId": 2, "name": 'Bayu', "supervisor": 1.0, "salary": 2000},
        ]),
        pd.DataFrame([
            {"empId": 1, "bonus": 2000},
            {"empId": 2, "bonus": 5000},
        ]),
    ), pd.DataFrame(columns=['name', 'bonus'])),

    ("Tabel bonus kosong", (
        pd.DataFrame([
            {"empId": 1, "name": 'Ana', "supervisor": None, "salary": 3000},
            {"empId": 2, "name": 'Bayu', "supervisor": 1.0, "salary": 2000},
        ]),
        pd.DataFrame({"empId": pd.Series(dtype="int64"), "bonus": pd.Series(dtype="int64")}),
    ), pd.DataFrame([
        {"name": 'Ana', "bonus": None},
        {"name": 'Bayu', "bonus": None},
    ])),

    ("Terdapat karyawan dengan bonus pada batas 1000", (
        pd.DataFrame([
            {"empId": 1, "name": 'Ana', "supervisor": None, "salary": 3000},
            {"empId": 2, "name": 'Bayu', "supervisor": 1.0, "salary": 2000},
            {"empId": 3, "name": 'Cindy', "supervisor": 1.0, "salary": 2500},
        ]),
        pd.DataFrame([
            {"empId": 1, "bonus": 1000},
            {"empId": 2, "bonus": 999},
        ]),
    ), pd.DataFrame([
        {"name": 'Bayu', "bonus": 999},
        {"name": 'Cindy', "bonus": None},
    ])),

]

run_tests(answer, cases)

## Soal 6
Diberikan sebuah tabel `Courses` dengan kolom `student` dan `class`. Ambil nama kelas yang memiliki minimal 5 siswa yang berbeda


### Data

In [ ]:
courses = pd.DataFrame({
    "student": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "A"],
    "class":   ["Math"]*6 + ["Biology", "Computer", "Math", "Math"],
})

display(courses)

### Solusi

In [ ]:
def answer(courses):
    per_kelas = (courses.groupby("class", as_index=False)["student"]
                        .nunique()
                        .rename(columns={"student": "n_siswa"}))

    # HAVING di SQL = mask yang diterapkan SESUDAH agregasi
    return per_kelas.loc[per_kelas["n_siswa"] >= 5, ["class"]]

### Hasil

In [ ]:
hasil = answer(courses)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 6
cases = [
    ("Dua kelas memenuhi syarat", (
        pd.DataFrame([
            {"student": 'A', "class": 'Math'},
            {"student": 'B', "class": 'Math'},
            {"student": 'C', "class": 'Math'},
            {"student": 'D', "class": 'Math'},
            {"student": 'E', "class": 'Math'},
            {"student": 'A', "class": 'Physics'},
            {"student": 'B', "class": 'Physics'},
            {"student": 'C', "class": 'Physics'},
            {"student": 'D', "class": 'Physics'},
            {"student": 'E', "class": 'Physics'},
            {"student": 'F', "class": 'Physics'},
        ]),
    ), pd.DataFrame([
        {"class": 'Math'},
        {"class": 'Physics'},
    ])),

    ("Ada siswa duplikat sehingga jumlah siswa unik kurang dari 5", (
        pd.DataFrame([
            {"student": 'A', "class": 'Math'},
            {"student": 'A', "class": 'Math'},
            {"student": 'B', "class": 'Math'},
            {"student": 'C', "class": 'Math'},
            {"student": 'D', "class": 'Math'},
            {"student": 'A', "class": 'Math'},
        ]),
    ), pd.DataFrame(columns=['class'])),

    ("Semua kelas memiliki siswa yang sedikit", (
        pd.DataFrame([
            {"student": 'A', "class": 'Art'},
            {"student": 'B', "class": 'Art'},
            {"student": 'C', "class": 'Biology'},
        ]),
    ), pd.DataFrame(columns=['class'])),

]

run_tests(answer, cases)

## Soal 7
Diberikan sebuah `Activity` dengan kolom `machine_id`, `process_id`, `activity_type`, `timestamp`. Kolom `activity_type` berisi `'start'` yang menandakan mulainya proses mesin dan `'end'` yang menandakan dihentikannya mesin. Hitung rata-rata durasi proses per mesin dan bulatkan dalam 3 angka desimal.


### Data

In [ ]:
activity = pd.DataFrame({
    "machine_id":    [0, 0, 0, 0, 1, 1, 1, 1],
    "process_id":    [0, 0, 1, 1, 0, 0, 1, 1],
    "activity_type": ["start", "end", "start", "end"] * 2,
    "timestamp":     [0.712, 1.520, 3.140, 4.120, 0.550, 1.550, 0.430, 1.420],
})

display(activity)

### Solusi

In [ ]:
def answer(activity):
    # long -> wide: 'start' dan 'end' menjadi dua kolom terpisah
    lebar = activity.pivot_table(index=["machine_id", "process_id"],
                                 columns="activity_type",
                                 values="timestamp",
                                 aggfunc="max").reset_index()
    lebar.columns.name = None
    lebar["durasi"] = lebar["end"] - lebar["start"]

    hasil = (lebar.groupby("machine_id", as_index=False)["durasi"]
                  .mean()
                  .rename(columns={"durasi": "processing_time"}))
    hasil["processing_time"] = hasil["processing_time"].round(3)
    return hasil

### Hasil

In [ ]:
hasil = answer(activity)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 7
cases = [
    ("Satu mesin dengan satu proses", (
        pd.DataFrame([
            {"machine_id": 0, "process_id": 0, "activity_type": 'start', "timestamp": 1.0},
            {"machine_id": 0, "process_id": 0, "activity_type": 'end', "timestamp": 2.5},
        ]),
    ), pd.DataFrame([
        {"machine_id": 0, "processing_time": 1.5},
    ])),

    ("Hasil harus dibulatkan ke 3 angka desimal", (
        pd.DataFrame([
            {"machine_id": 5, "process_id": 0, "activity_type": 'start', "timestamp": 0.0},
            {"machine_id": 5, "process_id": 0, "activity_type": 'end', "timestamp": 1.0},
            {"machine_id": 5, "process_id": 1, "activity_type": 'start', "timestamp": 0.0},
            {"machine_id": 5, "process_id": 1, "activity_type": 'end', "timestamp": 1.0004},
            {"machine_id": 5, "process_id": 2, "activity_type": 'start', "timestamp": 0.0},
            {"machine_id": 5, "process_id": 2, "activity_type": 'end', "timestamp": 1.0002},
        ]),
    ), pd.DataFrame([
        {"machine_id": 5, "processing_time": 1.0},
    ])),

    ("Urutan baris acak, 'end' bisa muncul sebelum 'start'", (
        pd.DataFrame([
            {"machine_id": 1, "process_id": 0, "activity_type": 'end', "timestamp": 4.0},
            {"machine_id": 2, "process_id": 0, "activity_type": 'end', "timestamp": 9.0},
            {"machine_id": 1, "process_id": 0, "activity_type": 'start', "timestamp": 1.0},
            {"machine_id": 2, "process_id": 0, "activity_type": 'start', "timestamp": 3.0},
        ]),
    ), pd.DataFrame([
        {"machine_id": 1, "processing_time": 3.0},
        {"machine_id": 2, "processing_time": 6.0},
    ])),

]

run_tests(answer, cases)

## Soal 8
Diberikan tiga tabel: 
- `Students` dengan kolom `student_id` dan `student_name`
- `Subjects` dengan kolom `subject_name`
- `Examinations` dengan kolom `student_id` dan `subject_name`

Hitung jumlah ujian yang diikuti setiap siswa untuk setiap mata pelajaran, termasuk mata pelajaran yang jumlah ujiannya 0


### Data

In [ ]:
students = pd.DataFrame({
    "student_id":   [1, 2, 13],
    "student_name": ["Alice", "Bob", "John"],
})
subjects = pd.DataFrame({"subject_name": ["Math", "Physics", "Programming"]})
examinations = pd.DataFrame({
    "student_id":   [1, 1, 1, 2, 1, 1, 13, 13, 13, 2],
    "subject_name": ["Math", "Physics", "Programming", "Programming",
                     "Physics", "Math", "Math", "Programming", "Physics", "Math"],
})

display(students)
display(subjects)
display(examinations)

### Solusi

In [ ]:
def answer(students, subjects, examinations):
    # 1) kerangka lengkap: setiap siswa x setiap mata pelajaran
    kerangka = students.merge(subjects, how="cross")

    # 2) hitung ujian yang benar-benar ada
    jumlah = (examinations.groupby(["student_id", "subject_name"], as_index=False)
                          .size()
                          .rename(columns={"size": "attended_exams"}))

    # 3) tempelkan ke kerangka, pasangan yang tidak ada diisi 0
    hasil = kerangka.merge(jumlah, on=["student_id", "subject_name"], how="left")
    hasil["attended_exams"] = hasil["attended_exams"].fillna(0).astype(int)
    return hasil.sort_values(["student_id", "subject_name"]).reset_index(drop=True)

### Hasil

In [ ]:
hasil = answer(students, subjects, examinations)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 8
cases = [
    ("Setiap siswa mengikuti semua mata pelajaran", (
        pd.DataFrame([
            {"student_id": 1, "student_name": 'Ana'},
        ]),
        pd.DataFrame([
            {"subject_name": 'Math'},
            {"subject_name": 'Physics'},
        ]),
        pd.DataFrame([
            {"student_id": 1, "subject_name": 'Math'},
            {"student_id": 1, "subject_name": 'Physics'},
        ]),
    ), pd.DataFrame([
        {"student_id": 1, "student_name": 'Ana', "subject_name": 'Math', "attended_exams": 1},
        {"student_id": 1, "student_name": 'Ana', "subject_name": 'Physics', "attended_exams": 1},
    ])),

    ("Tabel ujian kosong sehingga semua bernilai 0", (
        pd.DataFrame([
            {"student_id": 1, "student_name": 'Ana'},
            {"student_id": 2, "student_name": 'Budi'},
        ]),
        pd.DataFrame([
            {"subject_name": 'Math'},
        ]),
        pd.DataFrame({"student_id": pd.Series(dtype="int64"), "subject_name": pd.Series(dtype="str")}),
    ), pd.DataFrame([
        {"student_id": 1, "student_name": 'Ana', "subject_name": 'Math', "attended_exams": 0},
        {"student_id": 2, "student_name": 'Budi', "subject_name": 'Math', "attended_exams": 0},
    ])),

    ("Ada ujian berulang dan mata pelajaran yang tidak diikuti", (
        pd.DataFrame([
            {"student_id": 5, "student_name": 'Cita'},
        ]),
        pd.DataFrame([
            {"subject_name": 'Math'},
            {"subject_name": 'Seni'},
        ]),
        pd.DataFrame([
            {"student_id": 5, "subject_name": 'Math'},
            {"student_id": 5, "subject_name": 'Math'},
            {"student_id": 5, "subject_name": 'Math'},
        ]),
    ), pd.DataFrame([
        {"student_id": 5, "student_name": 'Cita', "subject_name": 'Math', "attended_exams": 3},
        {"student_id": 5, "student_name": 'Cita', "subject_name": 'Seni', "attended_exams": 0},
    ])),

]

run_tests(answer, cases)

## Soal 9
Diberikan sebuah tabel `Employee` dengan kolom `id`, `name`, `department`, dan `managerId`. Tampilkan nama manager yang setidaknya memiliki 5 bawahan secara langsung


### Data

In [ ]:
employee = pd.DataFrame({
    "id":         [101, 102, 103, 104, 105, 106, 107],
    "name":       ["John", "Dan", "James", "Amy", "Anne", "Ron", "Sara"],
    "department": ["A", "A", "A", "A", "A", "B", "A"],
    "managerId":  [np.nan, 101, 101, 101, 101, 101, 102],
})

display(employee)

### Solusi

In [ ]:
def answer(employee):
    # value_counts() otomatis mengabaikan NaN, jadi atasan teratas tidak ikut terhitung
    n_bawahan = employee["managerId"].value_counts()
    manajer_id = n_bawahan[n_bawahan >= 5].index
    return employee.loc[employee["id"].isin(manajer_id), ["name"]]

### Hasil

In [ ]:
hasil = answer(employee)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 9
cases = [
    ("Manager dengan tepat 5 bawahan langsung", (
        pd.DataFrame([
            {"id": 1, "name": 'Ana', "department": 'A', "managerId": np.nan},
            {"id": 2, "name": 'Budi', "department": 'A', "managerId": 1},
            {"id": 3, "name": 'Cita', "department": 'A', "managerId": 1},
            {"id": 4, "name": 'Deni', "department": 'A', "managerId": 1},
            {"id": 5, "name": 'Eka', "department": 'A', "managerId": 1},
            {"id": 6, "name": 'Fani', "department": 'B', "managerId": 1},
        ]),
    ), pd.DataFrame([
        {"name": 'Ana'},
    ])),

    ("Manager hanya memiliki 4 bawahan langsung", (
        pd.DataFrame([
            {"id": 1, "name": 'Ana', "department": 'A', "managerId": None},
            {"id": 2, "name": 'Budi', "department": 'A', "managerId": 1},
            {"id": 3, "name": 'Cita', "department": 'A', "managerId": 1},
            {"id": 4, "name": 'Deni', "department": 'A', "managerId": 1},
            {"id": 5, "name": 'Eka', "department": 'A', "managerId": 1},
        ]),
    ), pd.DataFrame(columns=['name'])),

    ("Bawahan tidak langsung tidak ikut dihitung", (
        pd.DataFrame([
            {"id": 1, "name": 'Ana', "department": 'A', "managerId": np.nan},
            {"id": 2, "name": 'Budi', "department": 'A', "managerId": 1},
            {"id": 3, "name": 'Cita', "department": 'A', "managerId": 1},
            {"id": 4, "name": 'Deni', "department": 'A', "managerId": 1},
            {"id": 5, "name": 'Eka', "department": 'A', "managerId": 1},
            {"id": 6, "name": 'Fani', "department": 'A', "managerId": 1},
            {"id": 7, "name": 'Gilang', "department": 'A', "managerId": 2},
            {"id": 8, "name": 'Hana', "department": 'A', "managerId": 2},
        ]),
    ), pd.DataFrame([
        {"name": 'Ana'},
    ])),

]

run_tests(answer, cases)

## Soal 10
Diberikan dua tabel: 
- `Signups` dengan kolom `user_id` dan `time_stamp`
- `Confirmations` dengan `user_id`, `time_stamp`, serta `action`
Hitung rasio konfirmasi setiap pengguna, dan bulatkan 2 desimal. Pengguna yang tidak pernah mencoba akan mendapatkan rasio 0.00


### Data

In [ ]:
signups = pd.DataFrame({
    "user_id":    [3, 7, 2, 6],
    "time_stamp": pd.to_datetime(["2020-03-21 10:16:13", "2020-01-04 13:57:59",
                                  "2020-07-29 23:09:44", "2020-12-09 10:39:37"]),
})
confirmations = pd.DataFrame({
    "user_id":    [3, 3, 7, 7, 7, 2],
    "time_stamp": pd.to_datetime(["2021-01-06 03:30:46", "2021-07-14 14:00:00",
                                  "2021-06-12 11:57:29", "2021-06-13 12:58:28",
                                  "2021-06-14 13:59:27", "2021-01-22 00:00:00"]),
    "action":     ["timeout", "timeout", "confirmed", "confirmed", "confirmed", "timeout"],
})

display(signups)
display(confirmations)

### Solusi

In [ ]:
def answer(signups, confirmations):
    # LEFT JOIN: pengguna tanpa catatan konfirmasi tetap muncul dengan action NaN
    gab = signups.merge(confirmations, on="user_id", how="left",
                        suffixes=("_signup", "_conf"))
    gab["is_confirmed"] = (gab["action"] == "confirmed").astype(int)

    hasil = (gab.groupby("user_id", as_index=False)["is_confirmed"]
                .mean()
                .rename(columns={"is_confirmed": "confirmation_rate"}))
    hasil["confirmation_rate"] = hasil["confirmation_rate"].round(2)
    return hasil

### Hasil

In [ ]:
hasil = answer(signups, confirmations)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 10
cases = [
    ("Semua percobaan berhasil dikonfirmasi", (
        pd.DataFrame([
            {"user_id": 1, "time_stamp": '2020-01-01 00:00:00'},
        ]),
        pd.DataFrame([
            {"user_id": 1, "time_stamp": '2020-01-02 00:00:00', "action": 'confirmed'},
            {"user_id": 1, "time_stamp": '2020-01-03 00:00:00', "action": 'confirmed'},
        ]),
    ), pd.DataFrame([
        {"user_id": 1, "confirmation_rate": 1.0},
    ])),

    ("Pengguna tanpa catatan konfirmasi mendapat rasio 0", (
        pd.DataFrame([
            {"user_id": 1, "time_stamp": '2020-01-01 00:00:00'},
            {"user_id": 2, "time_stamp": '2020-01-01 00:00:00'},
        ]),
        pd.DataFrame([
            {"user_id": 1, "time_stamp": '2020-01-02 00:00:00', "action": 'timeout'},
        ]),
    ), pd.DataFrame([
        {"user_id": 1, "confirmation_rate": 0.0},
        {"user_id": 2, "confirmation_rate": 0.0},
    ])),

    ("Rasio dibulatkan ke 2 angka desimal", (
        pd.DataFrame([
            {"user_id": 9, "time_stamp": '2020-01-01 00:00:00'},
        ]),
        pd.DataFrame([
            {"user_id": 9, "time_stamp": '2020-01-02 00:00:00', "action": 'confirmed'},
            {"user_id": 9, "time_stamp": '2020-01-03 00:00:00', "action": 'confirmed'},
            {"user_id": 9, "time_stamp": '2020-01-04 00:00:00', "action": 'timeout'},
        ]),
    ), pd.DataFrame([
        {"user_id": 9, "confirmation_rate": 0.67},
    ])),

]

run_tests(answer, cases)

## Soal 11
Diberikan dua tabel:
- `Prices` dengan kolom `product_id`, `start_date`, `end_date`, dan `price`
- `UnitsSold` dengan kolom `product_id`, `purchase_date`, dan `units`

Hitung weighted average dari harga jual setiap produk, yaitu total pendapatan dibagi total unit terjual, dan bulatkan 2 desimal. Sebuah penjualan memakai harga pada baris `Prices` yang rentang tanggalnya mencakup `purchase_date`. Produk yang tidak pernah terjual bernilai 0.


### Data

In [ ]:
prices = pd.DataFrame({
    "product_id": [1, 1, 2, 2, 3],
    "start_date": pd.to_datetime(["2019-02-17", "2019-03-01", "2019-02-01",
                                  "2019-02-21", "2019-01-01"]),
    "end_date":   pd.to_datetime(["2019-02-28", "2019-03-22", "2019-02-20",
                                  "2019-03-31", "2019-12-31"]),
    "price":      [5, 20, 15, 30, 99],
})
units_sold = pd.DataFrame({
    "product_id":    [1, 1, 2, 2],
    "purchase_date": pd.to_datetime(["2019-02-25", "2019-03-01",
                                     "2019-02-10", "2019-03-22"]),
    "units":         [100, 15, 200, 30],
})

display(prices)
display(units_sold)

### Solusi

In [ ]:
def answer(prices, units_sold):
    # pandas tidak punya join dengan kondisi rentang: join pada kunci kesetaraan dulu,
    # lalu saring hasilnya dengan kondisi tanggal sebagai mask
    kasar = prices.merge(units_sold, on="product_id", how="left")

    dalam_rentang = ((kasar["purchase_date"] >= kasar["start_date"]) &
                     (kasar["purchase_date"] <= kasar["end_date"]))
    # NaT dipertahankan supaya produk tanpa penjualan tidak ikut terbuang
    cocok = kasar[dalam_rentang | kasar["purchase_date"].isna()].copy()

    cocok["pendapatan"] = cocok["price"] * cocok["units"]
    agg = cocok.groupby("product_id", as_index=False).agg(
        total_pendapatan=("pendapatan", "sum"),
        total_unit=("units", "sum"),
    )

    # Produk yang punya penjualan tetapi semuanya di luar rentang ikut terbuang oleh
    # filter di atas, jadi tempelkan kembali ke daftar lengkap product_id
    semua_id = prices[["product_id"]].drop_duplicates()
    agg = semua_id.merge(agg, on="product_id", how="left")

    # total_unit kosong untuk produk tanpa penjualan -> hasil bagi NaN -> diisi 0
    agg["average_price"] = (agg["total_pendapatan"] / agg["total_unit"]).round(2).fillna(0)
    return agg[["product_id", "average_price"]]

### Hasil

In [ ]:
hasil = answer(prices, units_sold)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 11
cases = [
    ("Produk tanpa penjualan bernilai 0", (
        pd.DataFrame([
            {"product_id": 1, "start_date": pd.Timestamp("2019-01-01"), "end_date": pd.Timestamp("2019-01-31"), "price": 10},
        ]),
        pd.DataFrame({"product_id": pd.Series(dtype="int64"), "purchase_date": pd.Series(dtype="datetime64[us]"), "units": pd.Series(dtype="int64")}),
    ), pd.DataFrame([
        {"product_id": 1, "average_price": 0.0},
    ])),

    ("Penjualan tepat di batas rentang ikut dihitung", (
        pd.DataFrame([
            {"product_id": 1, "start_date": pd.Timestamp("2019-01-01"), "end_date": pd.Timestamp("2019-01-31"), "price": 10},
        ]),
        pd.DataFrame([
            {"product_id": 1, "purchase_date": pd.Timestamp("2019-01-01"), "units": 2},
            {"product_id": 1, "purchase_date": pd.Timestamp("2019-01-31"), "units": 3},
        ]),
    ), pd.DataFrame([
        {"product_id": 1, "average_price": 10.0},
    ])),

    ("Penjualan di luar rentang diabaikan", (
        pd.DataFrame([
            {"product_id": 1, "start_date": pd.Timestamp("2019-01-01"), "end_date": pd.Timestamp("2019-01-31"), "price": 10},
        ]),
        pd.DataFrame([
            {"product_id": 1, "purchase_date": pd.Timestamp("2019-02-01"), "units": 5},
        ]),
    ), pd.DataFrame([
        {"product_id": 1, "average_price": 0.0},
    ])),

]

run_tests(answer, cases)

## Soal 12
Diberikan sebuah tabel `Transactions` dengan kolom `id`, `country`, `state`, `amount`, dan `trans_date`. Kolom `state` berisi `'approved'` atau `'declined'`. Untuk setiap bulan dan negara, laporkan jumlah transaksi, jumlah transaksi yang disetujui, total nominal, dan total nominal yang disetujui. Negara yang bernilai kosong tetap dilaporkan sebagai satu kelompok tersendiri.


### Data

In [ ]:
transactions = pd.DataFrame({
    "id":         [121, 122, 123, 124, 125],
    "country":    ["US", "US", "US", "DE", None],
    "state":      ["approved", "declined", "approved", "approved", "approved"],
    "amount":     [1000, 2000, 2000, 2000, 500],
    "trans_date": pd.to_datetime(["2018-12-18", "2018-12-19", "2019-01-01",
                                  "2019-01-07", "2019-01-09"]),
})

display(transactions)

### Solusi

In [ ]:
def answer(transactions):
    t = transactions.copy()
    t["month"] = t["trans_date"].dt.strftime("%Y-%m")

    # SUM(CASE WHEN ...) di SQL = siapkan kolom bantu dulu, lalu jumlahkan biasa
    t["is_approved"] = (t["state"] == "approved").astype(int)
    t["approved_amount"] = t["amount"] * t["is_approved"]

    # dropna=False wajib, kalau tidak baris dengan country NaN akan hilang diam-diam
    return (t.groupby(["month", "country"], as_index=False, dropna=False)
             .agg(trans_count=("id", "size"),
                  approved_count=("is_approved", "sum"),
                  trans_total_amount=("amount", "sum"),
                  approved_total_amount=("approved_amount", "sum")))

### Hasil

In [ ]:
hasil = answer(transactions)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 12
cases = [
    ("Negara kosong tetap menjadi satu kelompok", (
        pd.DataFrame([
            {"id": 1, "country": None, "state": 'approved', "amount": 100, "trans_date": pd.Timestamp("2019-01-05")},
        ]),
    ), pd.DataFrame([
        {"month": '2019-01', "country": None, "trans_count": 1, "approved_count": 1, "trans_total_amount": 100, "approved_total_amount": 100},
    ])),

    ("Semua transaksi ditolak", (
        pd.DataFrame([
            {"id": 1, "country": 'US', "state": 'declined', "amount": 100, "trans_date": pd.Timestamp("2019-01-05")},
            {"id": 2, "country": 'US', "state": 'declined', "amount": 200, "trans_date": pd.Timestamp("2019-01-06")},
        ]),
    ), pd.DataFrame([
        {"month": '2019-01', "country": 'US', "trans_count": 2, "approved_count": 0, "trans_total_amount": 300, "approved_total_amount": 0},
    ])),

    ("Bulan berbeda dilaporkan terpisah", (
        pd.DataFrame([
            {"id": 1, "country": 'US', "state": 'approved', "amount": 100, "trans_date": pd.Timestamp("2019-01-31")},
            {"id": 2, "country": 'US', "state": 'approved', "amount": 200, "trans_date": pd.Timestamp("2019-02-01")},
        ]),
    ), pd.DataFrame([
        {"month": '2019-01', "country": 'US', "trans_count": 1, "approved_count": 1, "trans_total_amount": 100, "approved_total_amount": 100},
        {"month": '2019-02', "country": 'US', "trans_count": 1, "approved_count": 1, "trans_total_amount": 200, "approved_total_amount": 200},
    ])),

]

run_tests(answer, cases)

## Soal 13
Diberikan sebuah tabel `Delivery` dengan kolom `delivery_id`, `customer_id`, `order_date`, dan `customer_pref_delivery_date`. Pesanan disebut langsung bila tanggal pesan sama dengan tanggal preferensi pengiriman. Hitung persentase pesanan langsung dengan hanya melihat **pesanan pertama** setiap pelanggan, lalu bulatkan 2 desimal.


### Data

In [ ]:
delivery = pd.DataFrame({
    "delivery_id":                 [1, 2, 3, 4, 5, 6, 7],
    "customer_id":                 [1, 2, 1, 3, 3, 2, 4],
    "order_date":                  pd.to_datetime(
        ["2019-08-01", "2019-08-02", "2019-08-11", "2019-08-24",
         "2019-08-21", "2019-08-11", "2019-08-09"]),
    "customer_pref_delivery_date": pd.to_datetime(
        ["2019-08-02", "2019-08-02", "2019-08-12", "2019-08-24",
         "2019-08-22", "2019-08-13", "2019-08-09"]),
})

display(delivery)

### Solusi

In [ ]:
def answer(delivery):
    # idxmin() mengembalikan label baris dengan order_date terkecil per pelanggan
    pertama = delivery.loc[delivery.groupby("customer_id")["order_date"].idxmin()]

    langsung = pertama["order_date"] == pertama["customer_pref_delivery_date"]
    return pd.DataFrame({"immediate_percentage": [round(langsung.mean() * 100, 2)]})

### Hasil

In [ ]:
hasil = answer(delivery)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 13
cases = [
    ("Semua pesanan pertama bersifat langsung", (
        pd.DataFrame([
            {"delivery_id": 1, "customer_id": 1, "order_date": pd.Timestamp("2019-08-01"), "customer_pref_delivery_date": pd.Timestamp("2019-08-01")},
            {"delivery_id": 2, "customer_id": 2, "order_date": pd.Timestamp("2019-08-02"), "customer_pref_delivery_date": pd.Timestamp("2019-08-02")},
        ]),
    ), pd.DataFrame([
        {"immediate_percentage": 100.0},
    ])),

    ("Tidak ada pesanan pertama yang langsung", (
        pd.DataFrame([
            {"delivery_id": 1, "customer_id": 1, "order_date": pd.Timestamp("2019-08-01"), "customer_pref_delivery_date": pd.Timestamp("2019-08-03")},
            {"delivery_id": 2, "customer_id": 2, "order_date": pd.Timestamp("2019-08-02"), "customer_pref_delivery_date": pd.Timestamp("2019-08-05")},
        ]),
    ), pd.DataFrame([
        {"immediate_percentage": 0.0},
    ])),

    ("Pesanan pertama bukan baris pertama di tabel", (
        pd.DataFrame([
            {"delivery_id": 1, "customer_id": 1, "order_date": pd.Timestamp("2019-08-10"), "customer_pref_delivery_date": pd.Timestamp("2019-08-10")},
            {"delivery_id": 2, "customer_id": 1, "order_date": pd.Timestamp("2019-08-01"), "customer_pref_delivery_date": pd.Timestamp("2019-08-05")},
        ]),
    ), pd.DataFrame([
        {"immediate_percentage": 0.0},
    ])),

]

run_tests(answer, cases)

## Soal 14
Diberikan sebuah tabel `Activity` dengan kolom `player_id`, `device_id`, `event_date`, dan `games_played`. Hitung pecahan pemain yang login kembali **tepat sehari setelah** login pertamanya, lalu bulatkan 2 desimal.


### Data

In [ ]:
activity = pd.DataFrame({
    "player_id":    [1, 1, 2, 3, 3, 4],
    "device_id":    [2, 2, 3, 1, 4, 1],
    "event_date":   pd.to_datetime(["2016-03-01", "2016-03-02", "2017-06-25",
                                    "2016-03-02", "2018-07-03", "2018-07-03"]),
    "games_played": [5, 6, 1, 0, 5, 5],
})

display(activity)

### Solusi

In [ ]:
def answer(activity):
    pertama = (activity.groupby("player_id", as_index=False)["event_date"]
                       .min()
                       .rename(columns={"event_date": "login_pertama"}))
    pertama["hari_kedua"] = pertama["login_pertama"] + pd.Timedelta(days=1)

    # Kuncinya majemuk (pemain + tanggal), jadi isin() tidak cukup -> pakai merge
    kembali = activity.merge(pertama,
                             left_on=["player_id", "event_date"],
                             right_on=["player_id", "hari_kedua"],
                             how="inner")

    fraction = round(kembali["player_id"].nunique() / activity["player_id"].nunique(), 2)
    return pd.DataFrame({"fraction": [fraction]})

### Hasil

In [ ]:
hasil = answer(activity)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 14
cases = [
    ("Semua pemain kembali di hari kedua", (
        pd.DataFrame([
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-01"), "games_played": 1},
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-02"), "games_played": 1},
        ]),
    ), pd.DataFrame([
        {"fraction": 1.0},
    ])),

    ("Tidak ada pemain yang kembali", (
        pd.DataFrame([
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-01"), "games_played": 1},
            {"player_id": 2, "device_id": 1, "event_date": pd.Timestamp("2016-03-05"), "games_played": 1},
        ]),
    ), pd.DataFrame([
        {"fraction": 0.0},
    ])),

    ("Kembali tetapi bukan tepat sehari setelahnya", (
        pd.DataFrame([
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-01"), "games_played": 1},
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-02"), "games_played": 1},
            {"player_id": 2, "device_id": 1, "event_date": pd.Timestamp("2016-03-01"), "games_played": 1},
            {"player_id": 2, "device_id": 1, "event_date": pd.Timestamp("2016-03-03"), "games_played": 1},
        ]),
    ), pd.DataFrame([
        {"fraction": 0.5},
    ])),

]

run_tests(answer, cases)

## Soal 15
Diberikan sebuah tabel `Products` dengan kolom `product_id`, `new_price`, dan `change_date` yang mencatat riwayat perubahan harga. Tampilkan harga setiap produk pada tanggal **2019-08-16**. Produk yang belum pernah berubah harga sampai tanggal tersebut bernilai default 10.


### Data

In [ ]:
products = pd.DataFrame({
    "product_id":  [1, 2, 1, 1, 3],
    "new_price":   [20, 50, 30, 35, 20],
    "change_date": pd.to_datetime(["2019-08-14", "2019-08-14", "2019-08-15",
                                   "2019-08-16", "2019-08-18"]),
})

display(products)

### Solusi

In [ ]:
def answer(products):
    TANGGAL = pd.Timestamp("2019-08-16")

    # Ambil perubahan terakhir yang terjadi sampai dengan tanggal acuan
    riwayat = products[products["change_date"] <= TANGGAL]
    idx = riwayat.groupby("product_id")["change_date"].idxmax()
    terakhir = (riwayat.loc[idx, ["product_id", "new_price"]]
                       .rename(columns={"new_price": "price"}))

    # Produk yang belum pernah berubah harga tetap muncul dengan harga default
    semua_id = products[["product_id"]].drop_duplicates()
    hasil = semua_id.merge(terakhir, on="product_id", how="left")
    hasil["price"] = hasil["price"].fillna(10).astype(int)
    return hasil.sort_values("product_id").reset_index(drop=True)

### Hasil

In [ ]:
hasil = answer(products)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 15
cases = [
    ("Perubahan terakhir sebelum tanggal acuan yang dipakai", (
        pd.DataFrame([
            {"product_id": 1, "new_price": 20, "change_date": pd.Timestamp("2019-08-10")},
            {"product_id": 1, "new_price": 25, "change_date": pd.Timestamp("2019-08-12")},
        ]),
    ), pd.DataFrame([
        {"product_id": 1, "price": 25},
    ])),

    ("Perubahan sesudah tanggal acuan memakai harga default 10", (
        pd.DataFrame([
            {"product_id": 7, "new_price": 99, "change_date": pd.Timestamp("2019-08-20")},
        ]),
    ), pd.DataFrame([
        {"product_id": 7, "price": 10},
    ])),

    ("Perubahan tepat pada tanggal acuan ikut dihitung", (
        pd.DataFrame([
            {"product_id": 1, "new_price": 20, "change_date": pd.Timestamp("2019-08-15")},
            {"product_id": 1, "new_price": 40, "change_date": pd.Timestamp("2019-08-16")},
        ]),
    ), pd.DataFrame([
        {"product_id": 1, "price": 40},
    ])),

]

run_tests(answer, cases)

## Soal 16
Diberikan sebuah tabel `Accounts` dengan kolom `account_id` dan `income`. Hitung jumlah akun pada tiga kategori: `Low Salary` untuk pendapatan di bawah 20000, `Average Salary` untuk 20000 sampai 50000, dan `High Salary` untuk di atas 50000. Kategori yang tidak memiliki akun tetap muncul dengan nilai 0.


### Data

In [ ]:
accounts = pd.DataFrame({
    "account_id": [3, 2, 8, 6],
    "income":     [108939, 12747, 87709, 91796],
})

display(accounts)

### Solusi

In [ ]:
def answer(accounts):
    KATEGORI = ["Low Salary", "Average Salary", "High Salary"]

    # np.select menuliskan batas apa adanya: < 20000 eksklusif, <= 50000 inklusif.
    # pd.cut memaksa semua batas mengikuti satu arah, jadi tidak cocok di sini.
    kategori = np.select(
        [accounts["income"] < 20000, accounts["income"] <= 50000],
        ["Low Salary", "Average Salary"],
        default="High Salary",
    )

    # reindex inilah yang menjamin kategori tanpa akun tetap muncul
    return (pd.Series(kategori, dtype=object).value_counts()
            .reindex(KATEGORI, fill_value=0)
            .rename_axis("category")
            .reset_index(name="accounts_count"))

### Hasil

In [ ]:
hasil = answer(accounts)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 16
cases = [
    ("Ketiga kategori terisi", (
        pd.DataFrame([
            {"account_id": 1, "income": 10000},
            {"account_id": 2, "income": 30000},
            {"account_id": 3, "income": 90000},
        ]),
    ), pd.DataFrame([
        {"category": 'Low Salary', "accounts_count": 1},
        {"category": 'Average Salary', "accounts_count": 1},
        {"category": 'High Salary', "accounts_count": 1},
    ])),

    ("Tabel kosong tetap menampilkan tiga kategori", (
        pd.DataFrame({"account_id": pd.Series(dtype="int64"), "income": pd.Series(dtype="int64")}),
    ), pd.DataFrame([
        {"category": 'Low Salary', "accounts_count": 0},
        {"category": 'Average Salary', "accounts_count": 0},
        {"category": 'High Salary', "accounts_count": 0},
    ])),

    ("Nilai tepat pada batas 20000 dan 50000 masuk Average Salary", (
        pd.DataFrame([
            {"account_id": 1, "income": 20000},
            {"account_id": 2, "income": 50000},
            {"account_id": 3, "income": 50001},
        ]),
    ), pd.DataFrame([
        {"category": 'Low Salary', "accounts_count": 0},
        {"category": 'Average Salary', "accounts_count": 2},
        {"category": 'High Salary', "accounts_count": 1},
    ])),

]

run_tests(answer, cases)

## Soal 17
Diberikan sebuah tabel `Activities` dengan kolom `sell_date` dan `product`. Untuk setiap tanggal, laporkan jumlah produk unik yang terjual dan daftar nama produknya terurut alfabet serta dipisahkan koma.


### Data

In [ ]:
activities = pd.DataFrame({
    "sell_date": pd.to_datetime(["2020-05-30", "2020-06-01", "2020-06-02",
                                 "2020-05-30", "2020-06-01", "2020-06-02",
                                 "2020-05-30"]),
    "product":   ["Headphone", "Pencil", "Mask", "Basketball", "Bible", "Mask", "T-Shirt"],
})

display(activities)

### Solusi

In [ ]:
def answer(activities):
    # Agregasi menjadi string memang memerlukan lambda; tidak ada padanan bawaannya
    return (activities.groupby("sell_date")["product"]
            .agg(num_sold="nunique",
                 products=lambda s: ",".join(sorted(s.unique())))
            .reset_index()
            .sort_values("sell_date"))

### Hasil

In [ ]:
hasil = answer(activities)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 17
cases = [
    ("Produk yang sama pada satu tanggal hanya dihitung sekali", (
        pd.DataFrame([
            {"sell_date": pd.Timestamp("2020-06-01"), "product": 'Mask'},
            {"sell_date": pd.Timestamp("2020-06-01"), "product": 'Mask'},
        ]),
    ), pd.DataFrame([
        {"sell_date": pd.Timestamp("2020-06-01"), "num_sold": 1, "products": 'Mask'},
    ])),

    ("Nama produk diurutkan alfabet meski datanya acak", (
        pd.DataFrame([
            {"sell_date": pd.Timestamp("2020-06-01"), "product": 'Pencil'},
            {"sell_date": pd.Timestamp("2020-06-01"), "product": 'Bible'},
            {"sell_date": pd.Timestamp("2020-06-01"), "product": 'Mask'},
        ]),
    ), pd.DataFrame([
        {"sell_date": pd.Timestamp("2020-06-01"), "num_sold": 3, "products": 'Bible,Mask,Pencil'},
    ])),

    ("Tanggal berbeda dilaporkan terpisah", (
        pd.DataFrame([
            {"sell_date": pd.Timestamp("2020-06-01"), "product": 'Mask'},
            {"sell_date": pd.Timestamp("2020-06-02"), "product": 'Pencil'},
        ]),
    ), pd.DataFrame([
        {"sell_date": pd.Timestamp("2020-06-01"), "num_sold": 1, "products": 'Mask'},
        {"sell_date": pd.Timestamp("2020-06-02"), "num_sold": 1, "products": 'Pencil'},
    ])),

]

run_tests(answer, cases)

## Soal 18
Diberikan sebuah tabel `Queue` dengan kolom `person_id`, `person_name`, `weight`, dan `turn`. Sebuah bus berkapasitas 1000 kg dan penumpang naik sesuai urutan `turn`. Cari nama orang terakhir yang masih bisa naik tanpa melewati kapasitas bus.


### Data

In [ ]:
queue = pd.DataFrame({
    "person_id":   [5, 4, 3, 6, 1, 2],
    "person_name": ["Alice", "Bob", "Alex", "John Cena", "Winston", "Marie"],
    "weight":      [250, 175, 350, 400, 500, 200],
    "turn":        [5, 1, 2, 3, 6, 4],
})

display(queue)

### Solusi

In [ ]:
def answer(queue):
    # cumsum() hanya mengikuti urutan baris fisik, jadi sort_values WAJIB lebih dulu
    q = queue.sort_values("turn").reset_index(drop=True)
    q["total_kumulatif"] = q["weight"].cumsum()

    muat = q[q["total_kumulatif"] <= 1000]
    return pd.DataFrame({"person_name": [muat.iloc[-1]["person_name"]]})

### Hasil

In [ ]:
hasil = answer(queue)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 18
cases = [
    ("Data tidak terurut menurut turn", (
        pd.DataFrame([
            {"person_id": 1, "person_name": 'Ana', "weight": 900, "turn": 2},
            {"person_id": 2, "person_name": 'Budi', "weight": 100, "turn": 1},
        ]),
    ), pd.DataFrame([
        {"person_name": 'Ana'},
    ])),

    ("Semua penumpang muat", (
        pd.DataFrame([
            {"person_id": 1, "person_name": 'Ana', "weight": 100, "turn": 1},
            {"person_id": 2, "person_name": 'Budi', "weight": 200, "turn": 2},
        ]),
    ), pd.DataFrame([
        {"person_name": 'Budi'},
    ])),

    ("Berat tepat 1000 kg masih muat", (
        pd.DataFrame([
            {"person_id": 1, "person_name": 'Ana', "weight": 600, "turn": 1},
            {"person_id": 2, "person_name": 'Budi', "weight": 400, "turn": 2},
            {"person_id": 3, "person_name": 'Cita', "weight": 1, "turn": 3},
        ]),
    ), pd.DataFrame([
        {"person_name": 'Budi'},
    ])),

]

run_tests(answer, cases)

## Soal 19
Diberikan dua tabel:
- `Employee` dengan kolom `id`, `name`, `salary`, dan `departmentId`
- `Department` dengan kolom `id` dan `name`

Untuk setiap departemen, tampilkan karyawan yang gajinya termasuk **tiga nilai gaji tertinggi** di departemen tersebut. Bila beberapa karyawan bergaji sama, semuanya ikut ditampilkan.


### Data

In [ ]:
employee = pd.DataFrame({
    "id":           [1, 2, 3, 4, 5, 6, 7],
    "name":         ["Joe", "Henry", "Sam", "Max", "Janet", "Randy", "Will"],
    "salary":       [85000, 80000, 60000, 90000, 69000, 85000, 70000],
    "departmentId": [1, 2, 2, 1, 1, 1, 1],
})
department = pd.DataFrame({"id": [1, 2], "name": ["IT", "Sales"]})

display(employee)
display(department)

### Solusi

In [ ]:
def answer(employee, department):
    e = employee.copy()

    # method="dense" -> tiga NILAI gaji tertinggi, bukan tiga ORANG teratas.
    # "min" akan membuang gaji peringkat ketiga bila ada yang seri di atasnya.
    e["rk"] = e.groupby("departmentId")["salary"].rank(method="dense", ascending=False)
    teratas = e[e["rk"] <= 3]

    return (teratas.merge(
                department.rename(columns={"id": "departmentId", "name": "Department"}),
                on="departmentId", how="inner")
            .rename(columns={"name": "Employee", "salary": "Salary"})
            [["Department", "Employee", "Salary"]]
            .sort_values(["Department", "Salary"], ascending=[True, False])
            .reset_index(drop=True))

### Hasil

In [ ]:
hasil = answer(employee, department)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 19
cases = [
    ("Karyawan dengan gaji seri ikut semuanya", (
        pd.DataFrame([
            {"id": 1, "name": 'Ana', "salary": 100, "departmentId": 1},
            {"id": 2, "name": 'Budi', "salary": 100, "departmentId": 1},
        ]),
        pd.DataFrame([
            {"id": 1, "name": 'IT'},
        ]),
    ), pd.DataFrame([
        {"Department": 'IT', "Employee": 'Ana', "Salary": 100},
        {"Department": 'IT', "Employee": 'Budi', "Salary": 100},
    ])),

    ("Departemen dengan kurang dari tiga karyawan", (
        pd.DataFrame([
            {"id": 1, "name": 'Ana', "salary": 300, "departmentId": 1},
            {"id": 2, "name": 'Budi', "salary": 200, "departmentId": 1},
        ]),
        pd.DataFrame([
            {"id": 1, "name": 'IT'},
        ]),
    ), pd.DataFrame([
        {"Department": 'IT', "Employee": 'Ana', "Salary": 300},
        {"Department": 'IT', "Employee": 'Budi', "Salary": 200},
    ])),

    ("Gaji peringkat keempat tidak ikut", (
        pd.DataFrame([
            {"id": 1, "name": 'Ana', "salary": 400, "departmentId": 1},
            {"id": 2, "name": 'Budi', "salary": 300, "departmentId": 1},
            {"id": 3, "name": 'Cita', "salary": 200, "departmentId": 1},
            {"id": 4, "name": 'Deni', "salary": 100, "departmentId": 1},
        ]),
        pd.DataFrame([
            {"id": 1, "name": 'IT'},
        ]),
    ), pd.DataFrame([
        {"Department": 'IT', "Employee": 'Ana', "Salary": 400},
        {"Department": 'IT', "Employee": 'Budi', "Salary": 300},
        {"Department": 'IT', "Employee": 'Cita', "Salary": 200},
    ])),

]

run_tests(answer, cases)

## Soal 20
Diberikan sebuah tabel `Stadium` dengan kolom `id`, `visit_date`, dan `people`. Tampilkan baris yang menjadi bagian dari rentetan **tiga `id` berurutan atau lebih** yang semuanya memiliki `people` minimal 100.


### Data

In [ ]:
stadium = pd.DataFrame({
    "id":         [1, 2, 3, 4, 5, 6, 7, 8],
    "visit_date": pd.to_datetime(["2017-01-01", "2017-01-02", "2017-01-03", "2017-01-04",
                                  "2017-01-05", "2017-01-06", "2017-01-07", "2017-01-09"]),
    "people":     [10, 109, 150, 99, 145, 1455, 199, 188],
})

display(stadium)

### Solusi

In [ ]:
def answer(stadium):
    ramai = stadium[stadium["people"] >= 100].sort_values("id").reset_index(drop=True)

    # id - nomor baris: tetap selama id berurutan, melompat begitu ada yang bolong
    ramai["grup"] = ramai["id"] - np.arange(len(ramai))

    ukuran = ramai.groupby("grup")["id"].transform("size")
    return (ramai.loc[ukuran >= 3, ["id", "visit_date", "people"]]
                 .sort_values("visit_date")
                 .reset_index(drop=True))

### Hasil

In [ ]:
hasil = answer(stadium)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 20
cases = [
    ("Tepat tiga id berurutan", (
        pd.DataFrame([
            {"id": 1, "visit_date": pd.Timestamp("2017-01-01"), "people": 100},
            {"id": 2, "visit_date": pd.Timestamp("2017-01-02"), "people": 200},
            {"id": 3, "visit_date": pd.Timestamp("2017-01-03"), "people": 300},
        ]),
    ), pd.DataFrame([
        {"id": 1, "visit_date": pd.Timestamp("2017-01-01"), "people": 100},
        {"id": 2, "visit_date": pd.Timestamp("2017-01-02"), "people": 200},
        {"id": 3, "visit_date": pd.Timestamp("2017-01-03"), "people": 300},
    ])),

    ("Hanya dua id berurutan sehingga tidak memenuhi", (
        pd.DataFrame([
            {"id": 1, "visit_date": pd.Timestamp("2017-01-01"), "people": 100},
            {"id": 2, "visit_date": pd.Timestamp("2017-01-02"), "people": 200},
            {"id": 3, "visit_date": pd.Timestamp("2017-01-03"), "people": 10},
        ]),
    ), pd.DataFrame(columns=['id', 'visit_date', 'people'])),

    ("Rentetan pendek dibuang, rentetan panjang diambil", (
        pd.DataFrame([
            {"id": 1, "visit_date": pd.Timestamp("2017-01-01"), "people": 500},
            {"id": 2, "visit_date": pd.Timestamp("2017-01-02"), "people": 10},
            {"id": 3, "visit_date": pd.Timestamp("2017-01-03"), "people": 500},
            {"id": 4, "visit_date": pd.Timestamp("2017-01-04"), "people": 500},
            {"id": 5, "visit_date": pd.Timestamp("2017-01-05"), "people": 500},
        ]),
    ), pd.DataFrame([
        {"id": 3, "visit_date": pd.Timestamp("2017-01-03"), "people": 500},
        {"id": 4, "visit_date": pd.Timestamp("2017-01-04"), "people": 500},
        {"id": 5, "visit_date": pd.Timestamp("2017-01-05"), "people": 500},
    ])),

]

run_tests(answer, cases)

## Soal 21
Diberikan dua tabel:
- `Trips` dengan kolom `id`, `client_id`, `driver_id`, `city_id`, `status`, dan `request_at`
- `Users` dengan kolom `users_id`, `banned`, dan `role`

Untuk setiap tanggal antara 2013-10-01 dan 2013-10-03, hitung tingkat pembatalan yaitu jumlah perjalanan yang statusnya bukan `'completed'` dibagi total perjalanan, dibulatkan 2 desimal. Perjalanan yang penumpang atau pengemudinya diblokir tidak ikut dihitung.


### Data

In [ ]:
trips = pd.DataFrame({
    "id":         [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "client_id":  [1, 2, 3, 4, 1, 2, 3, 2, 3, 4],
    "driver_id":  [10, 11, 12, 13, 10, 11, 12, 12, 10, 13],
    "city_id":    [1, 1, 6, 6, 1, 6, 6, 12, 12, 12],
    "status":     ["completed", "cancelled_by_driver", "completed", "cancelled_by_client",
                   "completed", "completed", "completed", "completed",
                   "completed", "cancelled_by_driver"],
    "request_at": pd.to_datetime(["2013-10-01"] * 4 + ["2013-10-02"] * 3
                                 + ["2013-10-03"] * 3),
})
users = pd.DataFrame({
    "users_id": [1, 2, 3, 4, 10, 11, 12, 13],
    "banned":   ["No", "Yes", "No", "No", "No", "No", "No", "No"],
    "role":     ["client"] * 4 + ["driver"] * 4,
})

display(trips)
display(users)

### Solusi

In [ ]:
def answer(trips, users):
    # Users dipakai dua kali dengan peran berbeda; di pandas cukup satu himpunan + isin()
    tidak_diblokir = users.loc[users["banned"] == "No", "users_id"]

    # Saring dulu (memperkecil data), baru agregasi
    t = trips[trips["client_id"].isin(tidak_diblokir)
              & trips["driver_id"].isin(tidak_diblokir)]
    t = t[t["request_at"].between("2013-10-01", "2013-10-03")].copy()

    t["dibatalkan"] = t["status"] != "completed"
    hasil = (t.groupby("request_at", as_index=False)["dibatalkan"]
              .mean()
              .rename(columns={"request_at": "Day", "dibatalkan": "Cancellation Rate"}))
    hasil["Cancellation Rate"] = hasil["Cancellation Rate"].round(2)
    return hasil

### Hasil

In [ ]:
hasil = answer(trips, users)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 21
cases = [
    ("Semua perjalanan selesai", (
        pd.DataFrame([
            {"id": 1, "client_id": 1, "driver_id": 10, "city_id": 1, "status": 'completed', "request_at": pd.Timestamp("2013-10-01")},
        ]),
        pd.DataFrame([
            {"users_id": 1, "banned": 'No', "role": 'client'},
            {"users_id": 10, "banned": 'No', "role": 'driver'},
        ]),
    ), pd.DataFrame([
        {"Day": pd.Timestamp("2013-10-01"), "Cancellation Rate": 0.0},
    ])),

    ("Perjalanan dengan pengguna diblokir tidak dihitung", (
        pd.DataFrame([
            {"id": 1, "client_id": 1, "driver_id": 10, "city_id": 1, "status": 'cancelled_by_client', "request_at": pd.Timestamp("2013-10-01")},
            {"id": 2, "client_id": 2, "driver_id": 10, "city_id": 1, "status": 'completed', "request_at": pd.Timestamp("2013-10-01")},
        ]),
        pd.DataFrame([
            {"users_id": 1, "banned": 'Yes', "role": 'client'},
            {"users_id": 2, "banned": 'No', "role": 'client'},
            {"users_id": 10, "banned": 'No', "role": 'driver'},
        ]),
    ), pd.DataFrame([
        {"Day": pd.Timestamp("2013-10-01"), "Cancellation Rate": 0.0},
    ])),

    ("Perjalanan di luar rentang tanggal diabaikan", (
        pd.DataFrame([
            {"id": 1, "client_id": 1, "driver_id": 10, "city_id": 1, "status": 'cancelled_by_driver', "request_at": pd.Timestamp("2013-10-05")},
            {"id": 2, "client_id": 1, "driver_id": 10, "city_id": 1, "status": 'cancelled_by_driver', "request_at": pd.Timestamp("2013-10-02")},
        ]),
        pd.DataFrame([
            {"users_id": 1, "banned": 'No', "role": 'client'},
            {"users_id": 10, "banned": 'No', "role": 'driver'},
        ]),
    ), pd.DataFrame([
        {"Day": pd.Timestamp("2013-10-02"), "Cancellation Rate": 1.0},
    ])),

]

run_tests(answer, cases)

## Soal 22
Diberikan sebuah tabel `Customer` dengan kolom `customer_id`, `name`, `visited_on`, dan `amount`. Untuk setiap tanggal, hitung total belanja selama tujuh hari terakhir termasuk hari itu, beserta rata-rata hariannya yang dibulatkan 2 desimal. Baris baru muncul mulai hari ketujuh, yaitu saat jendela tujuh harinya sudah penuh.


### Data

In [ ]:
tanggal = pd.date_range("2019-01-01", periods=10, freq="D")
customer = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "name":        list("ABCDEFGHIJ"),
    "visited_on":  tanggal,
    "amount":      [100, 110, 120, 130, 110, 140, 150, 80, 110, 100],
})

display(customer)

### Solusi

In [ ]:
def answer(customer):
    # Satu tanggal bisa berisi beberapa transaksi -> agregasi harian dulu
    harian = (customer.groupby("visited_on", as_index=False)["amount"].sum()
                      .sort_values("visited_on")
                      .set_index("visited_on"))

    # "7D" = 7 hari kalender (butuh DatetimeIndex), bukan 7 baris seperti rolling(7)
    harian["total_7d"] = harian["amount"].rolling("7D").sum()

    mulai = harian.index.min() + pd.Timedelta(days=6)   # buang jendela yang belum penuh
    hasil = (harian.loc[harian.index >= mulai, ["total_7d"]]
                   .reset_index()
                   .rename(columns={"total_7d": "amount"}))
    hasil["average_amount"] = (hasil["amount"] / 7).round(2)
    return hasil

### Hasil

In [ ]:
hasil = answer(customer)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 22
cases = [
    ("Data kurang dari tujuh hari", (
        pd.DataFrame([
            {"customer_id": 1, "name": 'A', "visited_on": pd.Timestamp("2019-01-01"), "amount": 100},
            {"customer_id": 2, "name": 'B', "visited_on": pd.Timestamp("2019-01-02"), "amount": 100},
        ]),
    ), pd.DataFrame(columns=['visited_on', 'amount', 'average_amount'])),

    ("Tepat tujuh hari menghasilkan satu baris", (
        pd.DataFrame([
            {"customer_id": 1, "name": 'A', "visited_on": pd.Timestamp("2019-01-01"), "amount": 70},
            {"customer_id": 2, "name": 'A', "visited_on": pd.Timestamp("2019-01-02"), "amount": 70},
            {"customer_id": 3, "name": 'A', "visited_on": pd.Timestamp("2019-01-03"), "amount": 70},
            {"customer_id": 4, "name": 'A', "visited_on": pd.Timestamp("2019-01-04"), "amount": 70},
            {"customer_id": 5, "name": 'A', "visited_on": pd.Timestamp("2019-01-05"), "amount": 70},
            {"customer_id": 6, "name": 'A', "visited_on": pd.Timestamp("2019-01-06"), "amount": 70},
            {"customer_id": 7, "name": 'A', "visited_on": pd.Timestamp("2019-01-07"), "amount": 70},
        ]),
    ), pd.DataFrame([
        {"visited_on": pd.Timestamp("2019-01-07"), "amount": 490, "average_amount": 70.0},
    ])),

    ("Beberapa transaksi pada tanggal yang sama dijumlahkan dulu", (
        pd.DataFrame([
            {"customer_id": 1, "name": 'A', "visited_on": pd.Timestamp("2019-01-01"), "amount": 70},
            {"customer_id": 2, "name": 'A', "visited_on": pd.Timestamp("2019-01-02"), "amount": 70},
            {"customer_id": 3, "name": 'A', "visited_on": pd.Timestamp("2019-01-03"), "amount": 70},
            {"customer_id": 4, "name": 'A', "visited_on": pd.Timestamp("2019-01-04"), "amount": 70},
            {"customer_id": 5, "name": 'A', "visited_on": pd.Timestamp("2019-01-05"), "amount": 70},
            {"customer_id": 6, "name": 'A', "visited_on": pd.Timestamp("2019-01-06"), "amount": 70},
            {"customer_id": 7, "name": 'A', "visited_on": pd.Timestamp("2019-01-07"), "amount": 70},
            {"customer_id": 99, "name": 'Z', "visited_on": pd.Timestamp("2019-01-07"), "amount": 7},
        ]),
    ), pd.DataFrame([
        {"visited_on": pd.Timestamp("2019-01-07"), "amount": 497, "average_amount": 71.0},
    ])),

]

run_tests(answer, cases)

## Soal 23
Diberikan sebuah tabel `Activity` dengan kolom `player_id`, `device_id`, `event_date`, dan `games_played`. Tanggal instalasi seorang pemain adalah tanggal login pertamanya. Untuk setiap tanggal instalasi, laporkan jumlah pemain yang memasang pada hari itu dan pecahan pemain yang login kembali tepat sehari sesudahnya, dibulatkan 2 desimal.


### Data

In [ ]:
activity = pd.DataFrame({
    "player_id":    [1, 1, 2, 3, 3, 4, 4, 5],
    "device_id":    [2, 2, 3, 1, 4, 1, 1, 2],
    "event_date":   pd.to_datetime(["2016-03-01", "2016-03-02", "2017-06-25", "2016-03-01",
                                    "2016-03-02", "2016-03-01", "2016-03-05",
                                    "2017-06-25"]),
    "games_played": [5, 6, 1, 0, 5, 5, 2, 3],
})

display(activity)

### Solusi

In [ ]:
def answer(activity):
    kohort = (activity.groupby("player_id", as_index=False)["event_date"]
                      .min()
                      .rename(columns={"event_date": "install_date"}))
    kohort["hari_kedua"] = kohort["install_date"] + pd.Timedelta(days=1)

    # indicator=True memberi kolom _merge berisi both / left_only,
    # cara bersih untuk menandai "ketemu atau tidak"
    ditandai = kohort.merge(
        activity[["player_id", "event_date"]].drop_duplicates(),
        left_on=["player_id", "hari_kedua"],
        right_on=["player_id", "event_date"],
        how="left", indicator=True,
    )
    ditandai["kembali"] = (ditandai["_merge"] == "both").astype(int)

    hasil = (ditandai.groupby("install_date", as_index=False)
                     .agg(installs=("player_id", "nunique"),
                          Day1_retention=("kembali", "mean")))
    hasil["Day1_retention"] = hasil["Day1_retention"].round(2)
    return hasil

### Hasil

In [ ]:
hasil = answer(activity)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 23
cases = [
    ("Satu kohort dengan retensi penuh", (
        pd.DataFrame([
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-01"), "games_played": 1},
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-02"), "games_played": 1},
        ]),
    ), pd.DataFrame([
        {"install_date": pd.Timestamp("2016-03-01"), "installs": 1, "Day1_retention": 1.0},
    ])),

    ("Kohort tanpa satu pun pemain yang kembali", (
        pd.DataFrame([
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-01"), "games_played": 1},
            {"player_id": 2, "device_id": 1, "event_date": pd.Timestamp("2016-03-01"), "games_played": 1},
        ]),
    ), pd.DataFrame([
        {"install_date": pd.Timestamp("2016-03-01"), "installs": 2, "Day1_retention": 0.0},
    ])),

    ("Dua kohort dengan retensi berbeda", (
        pd.DataFrame([
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-01"), "games_played": 1},
            {"player_id": 1, "device_id": 1, "event_date": pd.Timestamp("2016-03-02"), "games_played": 1},
            {"player_id": 2, "device_id": 1, "event_date": pd.Timestamp("2016-03-01"), "games_played": 1},
            {"player_id": 3, "device_id": 1, "event_date": pd.Timestamp("2017-06-25"), "games_played": 1},
        ]),
    ), pd.DataFrame([
        {"install_date": pd.Timestamp("2016-03-01"), "installs": 2, "Day1_retention": 0.5},
        {"install_date": pd.Timestamp("2017-06-25"), "installs": 1, "Day1_retention": 0.0},
    ])),

]

run_tests(answer, cases)

## Soal 24
Diberikan dua tabel:
- `Items` dengan kolom `item_id`, `item_name`, dan `item_category`
- `Orders` dengan kolom `order_id`, `customer_id`, `order_date`, `item_id`, dan `quantity`

Buat tabel silang dengan baris berupa kategori barang dan kolom berupa tujuh hari dalam seminggu, berisi total kuantitas terjual. Setiap kategori dan setiap hari harus tetap muncul walaupun nilainya nol.


### Data

In [ ]:
items = pd.DataFrame({
    "item_id":       [1, 2, 3, 4],
    "item_name":     ["LC Alg. Book", "LC DB Book", "LC SmarthPhone", "LC Keychain"],
    "item_category": ["Book", "Book", "Phone", "Merch"],
})
orders = pd.DataFrame({
    "order_id":    [1, 2, 3, 4, 5, 6],
    "customer_id": [1, 1, 2, 3, 3, 2],
    "order_date":  pd.to_datetime(["2020-06-01", "2020-06-08", "2020-06-02",
                                   "2020-06-03", "2020-06-04", "2020-06-08"]),
    "item_id":     [1, 2, 1, 3, 3, 1],
    "quantity":    [10, 1, 5, 2, 3, 7],
})

display(items)
display(orders)

### Solusi

In [ ]:
def answer(items, orders):
    HARI = ["Monday", "Tuesday", "Wednesday", "Thursday",
            "Friday", "Saturday", "Sunday"]
    KATEGORI = sorted(items["item_category"].unique())

    gab = orders.merge(items, on="item_id", how="left")
    gab["hari"] = gab["order_date"].dt.day_name()

    pivot = gab.pivot_table(index="item_category", columns="hari",
                            values="quantity", aggfunc="sum", fill_value=0)

    # fill_value hanya menambal sel kosong di dalam kerangka yang sudah ada;
    # reindex-lah yang memunculkan baris & kolom yang sama sekali tidak ada
    hasil = (pivot.reindex(index=KATEGORI, columns=HARI, fill_value=0)
                  .fillna(0).astype(int)
                  .rename_axis("Category").reset_index())
    hasil.columns.name = None
    return hasil

### Hasil

In [ ]:
hasil = answer(items, orders)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 24
cases = [
    ("Kategori tanpa penjualan tetap muncul bernilai 0", (
        pd.DataFrame([
            {"item_id": 1, "item_name": 'Buku', "item_category": 'Book'},
            {"item_id": 2, "item_name": 'Gantungan', "item_category": 'Merch'},
        ]),
        pd.DataFrame([
            {"order_id": 1, "customer_id": 1, "order_date": pd.Timestamp("2020-06-01"), "item_id": 1, "quantity": 4},
        ]),
    ), pd.DataFrame([
        {"Category": 'Book', "Monday": 4, "Tuesday": 0, "Wednesday": 0, "Thursday": 0, "Friday": 0, "Saturday": 0, "Sunday": 0},
        {"Category": 'Merch', "Monday": 0, "Tuesday": 0, "Wednesday": 0, "Thursday": 0, "Friday": 0, "Saturday": 0, "Sunday": 0},
    ])),

    ("Beberapa pesanan pada hari yang sama dijumlahkan", (
        pd.DataFrame([
            {"item_id": 1, "item_name": 'Buku', "item_category": 'Book'},
        ]),
        pd.DataFrame([
            {"order_id": 1, "customer_id": 1, "order_date": pd.Timestamp("2020-06-01"), "item_id": 1, "quantity": 4},
            {"order_id": 2, "customer_id": 2, "order_date": pd.Timestamp("2020-06-08"), "item_id": 1, "quantity": 6},
        ]),
    ), pd.DataFrame([
        {"Category": 'Book', "Monday": 10, "Tuesday": 0, "Wednesday": 0, "Thursday": 0, "Friday": 0, "Saturday": 0, "Sunday": 0},
    ])),

    ("Pesanan akhir pekan masuk kolom yang benar", (
        pd.DataFrame([
            {"item_id": 1, "item_name": 'Buku', "item_category": 'Book'},
        ]),
        pd.DataFrame([
            {"order_id": 1, "customer_id": 1, "order_date": pd.Timestamp("2020-06-06"), "item_id": 1, "quantity": 2},
            {"order_id": 2, "customer_id": 1, "order_date": pd.Timestamp("2020-06-07"), "item_id": 1, "quantity": 3},
        ]),
    ), pd.DataFrame([
        {"Category": 'Book', "Monday": 0, "Tuesday": 0, "Wednesday": 0, "Thursday": 0, "Friday": 0, "Saturday": 2, "Sunday": 3},
    ])),

]

run_tests(answer, cases)

## Soal 25
Diberikan dua tabel:
- `Failed` dengan kolom `fail_date`
- `Succeeded` dengan kolom `success_date`

Keduanya mencatat status harian sebuah tugas terjadwal. Rangkum tanggal-tanggal pada tahun 2019 menjadi rentang berurutan yang statusnya sama, terurut menurut tanggal mulai.


### Data

In [ ]:
failed = pd.DataFrame({
    "fail_date": pd.to_datetime(["2018-12-28", "2018-12-29",
                                 "2019-01-04", "2019-01-05"]),
})
succeeded = pd.DataFrame({
    "success_date": pd.to_datetime(["2018-12-30", "2018-12-31", "2019-01-01",
                                    "2019-01-02", "2019-01-03", "2019-01-06"]),
})

display(failed)
display(succeeded)

### Solusi

In [ ]:
def answer(failed, succeeded):
    f = failed.rename(columns={"fail_date": "d"}).assign(period_state="failed")
    s = succeeded.rename(columns={"success_date": "d"}).assign(period_state="succeeded")

    semua = pd.concat([f, s], ignore_index=True)                    # UNION ALL
    semua = semua[semua["d"].between("2019-01-01", "2019-12-31")]
    semua = semua.sort_values("d").reset_index(drop=True)

    # cumcount() per status = ROW_NUMBER() OVER (PARTITION BY state ORDER BY d)
    semua["rn"] = semua.groupby("period_state").cumcount()
    semua["grup"] = semua["d"] - pd.to_timedelta(semua["rn"], unit="D")

    return (semua.groupby(["period_state", "grup"], as_index=False)
                 .agg(start_date=("d", "min"), end_date=("d", "max"))
                 .sort_values("start_date")
                 .drop(columns="grup")
                 .reset_index(drop=True))

### Hasil

In [ ]:
hasil = answer(failed, succeeded)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 25
cases = [
    ("Semua tanggal berstatus sama dan berurutan", (
        pd.DataFrame({"fail_date": pd.Series(dtype="datetime64[us]")}),
        pd.DataFrame([
            {"success_date": pd.Timestamp("2019-01-01")},
            {"success_date": pd.Timestamp("2019-01-02")},
            {"success_date": pd.Timestamp("2019-01-03")},
        ]),
    ), pd.DataFrame([
        {"period_state": 'succeeded', "start_date": pd.Timestamp("2019-01-01"), "end_date": pd.Timestamp("2019-01-03")},
    ])),

    ("Status berselang-seling sehingga tiap tanggal berdiri sendiri", (
        pd.DataFrame([
            {"fail_date": pd.Timestamp("2019-01-02")},
        ]),
        pd.DataFrame([
            {"success_date": pd.Timestamp("2019-01-01")},
            {"success_date": pd.Timestamp("2019-01-03")},
        ]),
    ), pd.DataFrame([
        {"period_state": 'succeeded', "start_date": pd.Timestamp("2019-01-01"), "end_date": pd.Timestamp("2019-01-01")},
        {"period_state": 'failed', "start_date": pd.Timestamp("2019-01-02"), "end_date": pd.Timestamp("2019-01-02")},
        {"period_state": 'succeeded', "start_date": pd.Timestamp("2019-01-03"), "end_date": pd.Timestamp("2019-01-03")},
    ])),

    ("Tanggal 2018 tidak ikut dilaporkan", (
        pd.DataFrame([
            {"fail_date": pd.Timestamp("2018-12-31")},
        ]),
        pd.DataFrame([
            {"success_date": pd.Timestamp("2019-01-01")},
            {"success_date": pd.Timestamp("2019-01-02")},
        ]),
    ), pd.DataFrame([
        {"period_state": 'succeeded', "start_date": pd.Timestamp("2019-01-01"), "end_date": pd.Timestamp("2019-01-02")},
    ])),

]

run_tests(answer, cases)

## Soal 26
Diberikan sebuah tabel `Numbers` dengan kolom `num` dan `frequency` yang menyimpan data dalam bentuk terkompresi, yaitu tiap nilai beserta berapa kali nilai itu muncul. Hitung median dari data yang diwakilinya, dibulatkan 1 desimal, tanpa memekarkan tabel menjadi daftar nilai satu per satu.


### Data

In [ ]:
numbers = pd.DataFrame({
    "num":       [0, 1, 2, 3],
    "frequency": [7, 1, 3, 1],
})

display(numbers)

### Solusi

In [ ]:
def answer(numbers):
    d = numbers.sort_values("num").reset_index(drop=True)
    d["kumulatif"] = d["frequency"].cumsum()
    total = int(d["frequency"].sum())

    # Rumus tunggal yang menangani total ganjil maupun genap
    p1, p2 = (total + 1) // 2, (total + 2) // 2

    def nilai_di_posisi(p):
        """Nilai pertama yang cumsum-nya sudah mencapai posisi p (mulai dari 1)."""
        return d.loc[d["kumulatif"] >= p, "num"].iloc[0]

    median = round((nilai_di_posisi(p1) + nilai_di_posisi(p2)) / 2, 1)
    return pd.DataFrame({"median": [median]})

### Hasil

In [ ]:
hasil = answer(numbers)
display(hasil)

### Test Case

In [ ]:
# 3 test case untuk soal 26
cases = [
    ("Jumlah data ganjil", (
        pd.DataFrame([
            {"num": 1, "frequency": 1},
            {"num": 2, "frequency": 1},
            {"num": 3, "frequency": 1},
        ]),
    ), pd.DataFrame([
        {"median": 2.0},
    ])),

    ("Jumlah data genap dengan dua nilai tengah berbeda", (
        pd.DataFrame([
            {"num": 1, "frequency": 1},
            {"num": 2, "frequency": 1},
        ]),
    ), pd.DataFrame([
        {"median": 1.5},
    ])),

    ("Satu nilai dengan frekuensi besar", (
        pd.DataFrame([
            {"num": 5, "frequency": 1000},
        ]),
    ), pd.DataFrame([
        {"median": 5.0},
    ])),

]

run_tests(answer, cases)